# <font color="steelblue">Enfermedad renal crónica (CKD)</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.

## <font color="steelblue">Objetivos del proyecto</font>

A partir de datos clínicos y analíticos, construir, comparar y **desplegar** un clasificador que prediga si un paciente padece **enfermedad renal crónica** (ERC). El dataset es **pequeño (400 registros)**, de **tipos mixtos** (11 numéricas, 14 nominales) y con **muchos valores faltantes** y "sucios": por eso el foco del proyecto está en hacer las cosas **con rigor**. Al terminar, debéis ser capaces de:

* **Limpiar** datos reales sucios (tabuladores, `?`, tipos mal leídos) y **analizar la estructura de los faltantes**.
* Diseñar una **estrategia de imputación** correcta y **sin fuga de datos** (dentro de `Pipeline`).
* **Comparar** varias familias de clasificadores con una **validación robusta para muestras pequeñas**.
* Tratar el (moderado) **desequilibrio** con **ponderación de muestras**/remuestreo.
* **Optimizar hiperparámetros** (con CV anidada), **combinar modelos** si aporta, buscar **parsimonia**, **interpretar** y **desplegar**.


## <font color="steelblue">El conjunto de datos</font>

### <font color="steelblue">Origen y estructura</font>

El conjunto procede del **UCI Machine Learning Repository** (id 336) y recoge la información clínica de **400 pacientes** atendidos en un hospital de **Karaikudi (Tamil Nadu, India)** durante un período de aproximadamente **dos meses**. Cada registro describe a un paciente mediante **24 predictoras** —de las cuales 11 son numéricas y 14 nominales— y una **variable objetivo binaria** que indica si padece o no enfermedad renal crónica. Las edades abarcan desde los 2 hasta los 90 años, de modo que la muestra incluye tanto población pediátrica como geriátrica.

Las predictoras se agrupan de forma natural según la **prueba diagnóstica** de la que proceden: exploración básica, análisis de orina, análisis de sangre e historia clínica. Esa agrupación no es casual, y conviene tenerla presente porque determina qué variables están disponibles en cada momento de la asistencia.

### <font color="steelblue">Diccionario de variables</font>

**Bloque 1 — Demografía y exploración básica**

| Variable | Tipo | Unidad | Descripción |
|---|---|---|---|
| `age` | Numérica | años | Edad del paciente (rango observado: 2–90). |
| `bp` | Numérica | mm/Hg | **Presión arterial**. Los valores registrados (60, 70, 80…) sugieren que se trata de la presión **diastólica**, no de la sistólica. |

**Bloque 2 — Análisis de orina**

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `sg` | **Ordinal** | 1.005, 1.010, 1.015, 1.020, 1.025 | **Gravedad específica** de la orina: mide su concentración. Un valor **bajo** indica que el riñón ha perdido la capacidad de concentrar la orina, un signo característico de la enfermedad. |
| `al` | **Ordinal** | 0–5 | **Albúmina** en orina. Un valor **alto** revela que el riñón deja escapar proteínas que debería retener (albuminuria). |
| `su` | **Ordinal** | 0–5 | **Azúcar** en orina (glucosuria). |
| `rbc` | Nominal | normal / abnormal | **Hematíes** (glóbulos rojos) en el sedimento urinario. |
| `pc` | Nominal | normal / abnormal | **Células de pus** (leucocitos) en orina. |
| `pcc` | Nominal | present / notpresent | **Cúmulos de células de pus**, indicativos de infección o inflamación. |
| `ba` | Nominal | present / notpresent | **Bacterias** en orina. |

**Bloque 3 — Análisis de sangre**

| Variable | Unidad | Descripción |
|---|---|---|
| `bgr` | mg/dl | **Glucosa en sangre**, medida aleatoria (*random*), sin ayuno previo. |
| `bu` | mg/dl | **Urea** en sangre. Producto de desecho que el riñón elimina; se acumula cuando falla. |
| `sc` | mg/dl | **Creatinina sérica**. Es el **marcador clave**: se eleva cuando el filtrado glomerular disminuye. |
| `sod` | mEq/L | **Sodio** sérico. |
| `pot` | mEq/L | **Potasio** sérico. Su acumulación es una complicación grave de la insuficiencia renal. |
| `hemo` | g/dl | **Hemoglobina**. Suele estar **baja** en la enfermedad renal, porque el riñón enfermo produce menos eritropoyetina. |
| `pcv` | % | **Volumen corpuscular** (hematocrito): porcentaje del volumen sanguíneo ocupado por glóbulos rojos. |
| `wbcc` | células/mm³ | **Recuento de leucocitos** (glóbulos blancos). En el fichero original figura como `wc`. |
| `rbcc` | millones/mm³ | **Recuento de hematíes**. En el fichero original, `rc`. |

**Bloque 4 — Historia clínica y comorbilidades**

| Variable | Valores | Descripción |
|---|---|---|
| `htn` | yes / no | **Hipertensión** arterial diagnosticada. |
| `dm` | yes / no | **Diabetes mellitus**. Junto con la hipertensión, es la principal causa de enfermedad renal crónica. |
| `cad` | yes / no | **Enfermedad arterial coronaria**. |
| `appet` | good / poor | **Apetito** del paciente. |
| `pe` | yes / no | **Edema en los pies** (*pedal edema*): retención de líquidos. |
| `ane` | yes / no | **Anemia**. |

**Variable objetivo**

| Variable | Valores | Descripción |
|---|---|---|
| `class` | `ckd` / `notckd` | Presencia (`ckd`) o ausencia (`notckd`) de **enfermedad renal crónica**. Reparto: **250 casos con ERC y 150 sin ella**, un desequilibrio moderado y manejable. |

> **Avisos de calidad (hay que tratarlos antes de modelar):**
> * Muchos **valores faltantes** repartidos por casi todas las columnas. Las más afectadas son `rbc` y `pc`, con alrededor de un **38 %** de ausencias.
> * Datos **sucios**: espacios/tabuladores (`'ckd\t'`, `'\t43'`), lo que hace que columnas numéricas se lean como **texto**. Ojo también con `dm` y `cad`, que presentan variantes como `'\tyes'` o `' yes'` que pandas trata como **categorías distintas**.
> * Los faltantes se codifican con `'?'`; conviene cargarlos con `na_values=['?', '\t?']` y aplicar un `.str.strip()` a todas las columnas de texto **antes** de convertir tipos.
> * Las nominales vienen como **cadenas** (`normal/abnormal`, `present/notpresent`, `yes/no`, `good/poor`).
> * `sg`, `al`, `su` son **categóricas ordinales** (no continuas), aunque en la práctica suelen tratarse como numéricas por su carácter graduado.

### <font color="steelblue">Advertencias metodológicas</font>

1. **Circularidad diagnóstica: el aviso más importante.** La enfermedad renal crónica **se define clínicamente** a partir del filtrado glomerular —que se calcula justamente con la **creatinina sérica** (`sc`)— y de la **albuminuria** (`al`). Es decir, estas variables no son *predictores* de la etiqueta: son, en buena medida, **los criterios con los que se asignó la etiqueta**. Predecir `class` a partir de `sc` y `al` equivale a predecir un diagnóstico a partir de su propia definición. Esto explica que casi cualquier modelo alcance sobre estos datos una exactitud próxima al 100 %, y que ese resultado **no demuestre nada**. Un proyecto interesante consiste precisamente en **excluir `sc` y `al`** y comprobar cuánta señal queda en el resto.

2. **Solo 400 pacientes y 24 variables.** Con una muestra tan reducida, las estimaciones de rendimiento tienen **mucha varianza**: una única partición entrenamiento/test puede variar varios puntos según la semilla. La **validación cruzada repetida** no es un lujo, es un requisito, y conviene reportar la desviación típica junto a la media.

3. **Los faltantes pueden no ser aleatorios.** Merece la pena comprobar si la proporción de ausencias difiere entre las clases `ckd` y `notckd`: si a los pacientes sanos no se les solicitaron ciertas pruebas, la **propia ausencia del dato sería informativa** y un imputador simple estaría, de hecho, filtrando la respuesta. Un `df.isna().groupby(df['class']).mean()` responde a esta pregunta en una línea.

4. **Coherencia clínica de los signos.** Las asociaciones esperadas son: hemoglobina, hematocrito y recuento de hematíes **bajos**; gravedad específica **baja**; albúmina y creatinina **altas**. Comprobar que el modelo aprende esas direcciones (con valores SHAP, por ejemplo) es una excelente **validación de sentido común**: si el modelo dice que una hemoglobina alta favorece el diagnóstico de ERC, algo va mal.

5. **Redundancia entre variables.** `hemo`, `pcv` y `rbcc` miden esencialmente lo mismo (la masa de glóbulos rojos), al igual que `ane` la resume de forma binaria; `bu` y `sc` comparten el mismo mecanismo. Esta **multicolinealidad** no perjudica a los árboles, pero reparte arbitrariamente la importancia entre variables intercambiables.

6. **Validez externa nula.** Un único hospital, dos meses de recogida y 400 pacientes con edades entre 2 y 90 años no constituyen una muestra representativa de ninguna población. El modelo resultante es un **ejercicio didáctico**, no una herramienta diagnóstica.

## <font color="steelblue">Reglas del juego (buenas prácticas obligatorias)</font>

1. **Partición primero**, estratificada por la clase; el *test* solo se toca al final.
2. **Imputación, codificación, escalado y remuestreo SIN FUGA:** se ajustan **solo con el *train***, dentro de un **`Pipeline`**, y se rehacen en **cada pliegue** de la CV. *(Imputar con la media de todo el dataset antes de partir es una fuga.)*
3. **Validación robusta para n pequeño:** con 400 registros, un único 80/20 deja ~80 casos de test (ruidoso). Apoyaos en **validación cruzada repetida** para comparar y **CV anidada** para estimar el rendimiento al optimizar.
4. **El equilibrado solo en *train*** (material 11). El *test* conserva la proporción real.
5. **Desconfiad de la perfección.** Es un dataset **muy separable**: ver *accuracy* ~99–100 % es normal, pero **comprobad que no hay fuga** (ver Fase 7) y valorad la **parsimonia** (¿pocas variables bastan?).
6. **Reproducibilidad y honestidad:** `random_state` fijado; reportad lo que no funcionó.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

In [ ]:
# !pip -q install ucimlrepo imbalanced-learn gradio optuna scikit-learn shap
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
RNG = 42

In [ ]:
# Paso 1: descargar el dataset
from ucimlrepo import fetch_ucirepo
chronic_kidney_disease = fetch_ucirepo(id=336)
ckd_features = pd.DataFrame(chronic_kidney_disease.data.features)
ckd_target = pd.DataFrame(chronic_kidney_disease.data.target)
ckd = pd.concat([ckd_features, ckd_target], axis=1)
ckd.head()

# <font color="steelblue">Fase 1 — Comprensión, calidad de datos y EDA</font>

**Tareas obligatorias**
1. **Inspección de tipos.** ¿Qué columnas numéricas se han leído como **texto**? ¿Por qué? Mirad `ckd.dtypes` y valores únicos de las sospechosas.
2. **Mapa de faltantes.** Porcentaje de `NaN` por columna y, sobre todo, **patrón**: ¿hay filas con muchos huecos? ¿La cantidad de faltantes **difiere entre clases**? *(Guardad esta observación para la Fase 7.)*
3. **Distribución del objetivo** (`ckd`/`notckd`) y de las variables clave (`sc`, `hemo`, `al`, `sg`, `htn`, `dm`).
4. **Relación con el objetivo:** distribución de creatinina, hemoglobina y albúmina por clase; tasa de ERC según hipertensión/diabetes.
5. **Conclusión:** 3–4 hallazgos y qué variables parecen más discriminantes.

> **A responder:** ¿qué métricas usaréis? Con clases 250/150 la *accuracy* es algo más informativa que en casos muy desequilibrados, pero seguid mirando **sensibilidad/recall** y **F1**.

# <font color="steelblue">Fase 2 — Limpieza, codificación, imputación e particionado</font>

Este es el **núcleo metodológico** del proyecto. Trabajad con cuidado.

**2A. Limpieza y codificación (obligatorio)**
1. **Quitad espacios/tabuladores** de las columnas de texto (`.str.strip()`).
2. **Forzad a numérico** las columnas que deberían serlo (`pd.to_numeric(..., errors='coerce')`); los valores ilegibles pasan a `NaN`.
3. **Codificad** el objetivo (`ckd`→1, `notckd`→0) y las binarias (`normal/abnormal`, `present/notpresent`, `yes/no`, `good/poor` → 0/1). Tratad `sg`, `al`, `su` como **ordinales**.

**2B. Partición (obligatorio, ¡antes de imputar!)**
4. `X`/`y` y **partición estratificada** *train*/*test*.

**2C. Imputación SIN FUGA (el reto central)**
5. Comparad **al menos dos** estrategias de imputación, siempre **dentro del `Pipeline`** (se ajustan solo con el *train* de cada pliegue):
   * `SimpleImputer` (mediana para numéricas, moda para categóricas),
   * `KNNImputer`,
   * (extra) `IterativeImputer` (MICE) o imputación basada en **Random Forest**.
6. Encapsulad imputación + escalado (donde haga falta) + modelo en un **`Pipeline`**.

> **Error clásico (penalizado):** imputar/escalar sobre **todo** el dataset antes de partir. Eso filtra información del *test* al *train*.

# <font color="steelblue">Fase 3 — Modelos base y comparación</font>

**Tareas obligatorias**
1. Comparad **≥5 familias** del curso: **Regresión logística**, **kNN**, **SVM**, **Árbol**, **Random Forest**, **HistGradientBoosting** (o XGBoost/LightGBM/CatBoost), **Naive Bayes**.
2. **Validación robusta para n pequeño:** usad **`RepeatedStratifiedKFold`** (p. ej. 5×10) y reportad media ± desviación. Un solo *split* no es fiable con 400 casos.
3. **Tabla** comparativa (F1 y/o exactitud balanceada) y comentario.

> Aquí el coste de cómputo **no** es problema (dataset pequeño): aprovechad para hacer validación **repetida**.

# <font color="steelblue">Fase 4 — Ponderación de muestras y equilibrado</font>

El desequilibrio aquí es **moderado** (~250/150), pero sigue siendo obligatorio **evaluar** el efecto de tratarlo (material **11**). Sobre los 2–3 mejores modelos:

1. **Sin tratamiento** (línea base).
2. **Sensible al coste:** `class_weight='balanced'`.
3. **Sobremuestreo:** **SMOTE** (recordad: SMOTE necesita datos **ya imputados y numéricos**, así que va **después** de la imputación dentro del `ImbPipeline`; para variables categóricas considerad **SMOTENC**).
4. (Opcional) Submuestreo/híbrido.

Reportad **recall** de la clase de interés, **F1** y exactitud balanceada, y comentad si con este equilibrio moderado el tratamiento **aporta o no**.

> **Sin fugas:** remuestreo **dentro** de un `Pipeline` de *imbalanced-learn*, solo en *train*.

# <font color="steelblue">Fase 5 — Optimización de hiperparámetros</font>

**Tareas obligatorias**
1. Optimizad los **2–3 mejores** (modelo + equilibrado).
2. `GridSearchCV`/`RandomizedSearchCV`/**Optuna**, con CV estratificada y la misma métrica; búsqueda **sobre el `Pipeline`** (prefijo `clf__`, `imp__`, etc.).
3. **CV anidada (recomendada por el n pequeño):** estimad el rendimiento con un bucle externo de CV y la búsqueda en el interno, para no ser optimistas. Comparad con la estimación simple.
4. Reportad mejores hiperparámetros y mejora frente a defaults.

> Como el dataset es pequeño, una **rejilla** exhaustiva es perfectamente viable aquí.

# <font color="steelblue">Fase 6 — Combinación de modelos</font>

1. Combinad los mejores modelos con **`VotingClassifier`** (votación **blanda**) y/o **`StackingClassifier`**.
2. Comparad frente al **mejor individual**: con un problema tan separable, **quizá no mejore** (o ya esté en el techo). Eso también es un resultado válido.
3. **Combinad solo si aporta** mejora real (requisito: *si fuera necesario*).

# <font color="steelblue">Fase 7 — Evaluación, parsimonia e interpretación</font>

**Tareas obligatorias** (el *test* se usa una sola vez)
1. Evaluad el **modelo final** en el *test*: **matriz de confusión**, `classification_report`, **F1**, **recall**, **ROC-AUC**. *(Con tan pocos casos de test, acompañad con la estimación por CV.)*
2. **Parsimonia / selección de variables.** ¿Cuántas variables hacen falta para casi el mismo rendimiento? Probad selección por importancia o **Chi-Cuadrado** (como en la literatura) y comparad un modelo **reducido** (p. ej. 5–8 variables: `sc`, `hemo`, `al`, `sg`, `htn`, `dm`…) con el completo.
3. **Interpretabilidad:** `permutation_importance` y/o **SHAP**. ¿Coincide con la clínica (creatinina alta, hemoglobina baja, albuminuria)?
4. **La trampa del 100 %.** Si el modelo es casi perfecto, **comprobad que no es por fuga**: en particular, ¿el **patrón de faltantes** depende de la clase (p. ej. a los `notckd` se les hicieron menos análisis)? Si imputar/usar indicadores de "faltante" mete información del desenlace, discutidlo.
5. **Discusión crítica:** tamaño muestral, origen único (un hospital), generalización.

# <font color="steelblue">Fase 8 — Despliegue del modelo</font>

1. **Persistencia:** guardad el **`Pipeline` completo** (imputación + preprocesado + modelo) con `joblib`; debe aceptar datos **crudos con faltantes** y devolver la predicción.
2. **Función de predicción:** `predecir_ckd(...)` que tome los valores del paciente (admitiendo huecos) y devuelva clase y **probabilidad**.
3. **Interfaz interactiva:** app con **Gradio** (o `ipywidgets`) con los campos clínicos principales; en Colab genera un **enlace público** (incluidlo en la entrega).
4. (Opcional, nota extra) **Streamlit**/**FastAPI** (como el *ML-CKDP* de la literatura).

> **Aviso clínico (obligatorio en la interfaz):** herramienta **educativa**, no es un diagnóstico médico.

# <font color="steelblue">Pistas y errores típicos</font>

* **Limpia antes de nada.** Si no quitas tabuladores ni fuerzas a numérico, columnas como `pcv`/`wbcc`/`rbcc` quedan como texto y todo falla silenciosamente.
* **Imputa dentro del `Pipeline`.** Imputar con la media global antes de partir es la fuga más común en este dataset.
* **n pequeño = varianza alta.** No te fíes de un único 80/20; usa **CV repetida** y, al optimizar, **CV anidada**.
* **¿100 % de acierto?** Es plausible (dataset muy separable), pero **sospecha**: revisa si el **patrón de faltantes** delata la clase y si tu modelo "acierta" por eso.
* **Parsimonia:** un modelo con 5–6 variables clínicas puede igualar al completo y es **más interpretable y desplegable**.
* **Despliegue:** guarda el **Pipeline entero** para que la app tolere entradas con huecos.

# <font color="steelblue">Referencias</font>

* Rubini, L., Soundarapandian, P. & Eswaran, P. (2015). *Chronic Kidney Disease*. UCI ML Repository (id 336).
* *Chronic kidney disease prediction based on machine learning algorithms*. PMC, 2023.
* *ML-CKDP: Machine learning-based CKD prediction with smart web application*. PMC, 2024.
* Levey, A. S. & Coresh, J. (2012). *Chronic kidney disease*. The Lancet, 379, 165–180.
* Cuadernos del curso: *Equilibrando las muestras*, *Random Forest*, *Boosting*, *Regresión logística binaria*.